# Week 14: LLM Fine-Tuning & Quantization
## Notebook 11: GGUF Format & Block-Level Quantization (k-quants)

### 1. Conceptual Intuition & Theoretical Background
Running 70B parameter models on consumer hardware (MacBooks, commodity CPUs) is made possible by **GGUF (Georgi Gerganov Universal Format)** and **llama.cpp**.

**GGUF Design Principles:**
1. **Single-File Encapsulation:** Contains model architecture metadata, tokenizer vocabulary, and quantized tensor blocks in a single memory-mappable binary.
2. **Zero-Allocation Memory Mapping (`mmap`):** Loads model files directly from disk into virtual address space without double allocation in RAM.
3. **Block-Level Quantization (k-quants):**
   Instead of using a single scale for an entire weight matrix, weights are partitioned into small blocks of 32 or 256 numbers:
$$W_i = d \cdot q_i + m$$
   Where $d$ is a block scale factor, $m$ is an offset minimum, and $q_i$ is a low-bit integer ($2, 4, \dots, 8$ bits).

---

### 2. Scratch Python Implementation: Block Quantization Engine (Q4_0)

In [ ]:
import numpy as np

def quantize_block_q4_0(weights_block):
    """
    Simulates llama.cpp Q4_0 block quantization on a block of 32 floats.
    Scale: d = max(|W|) / -8
    Quantized values: 4-bit signed integers in [-8, 7]
    """
    block = np.asarray(weights_block, dtype=np.float32)
    assert len(block) == 32
    
    # 1. Compute block scale d (FP16 in reality)
    amax = np.max(np.abs(block))
    d = amax / 8.0 if amax > 0 else 1.0
    
    # 2. Quantize each element to 4-bit signed int [-8, 7]
    q = np.clip(np.round(block / d), -8, 7).astype(np.int8)
    
    # 3. Simulate binary packing: 2 4-bit values per byte
    # Low nibble and high nibble
    packed_bytes = bytearray()
    for i in range(0, 32, 2):
        low = q[i] & 0x0F
        high = (q[i + 1] & 0x0F) << 4
        packed_bytes.append(low | high)
        
    return d, packed_bytes

def dequantize_block_q4_0(d, packed_bytes):
    """Dequantizes packed 4-bit bytes back to 32 floats."""
    unpacked_q = []
    for byte in packed_bytes:
        # Extract low nibble (signed 4-bit)
        low = byte & 0x0F
        if low >= 8: low -= 16 # Sign extend
        # Extract high nibble
        high = (byte >> 4) & 0x0F
        if high >= 8: high -= 16
        unpacked_q.extend([low, high])
        
    return np.array(unpacked_q, dtype=np.float32) * d

# Test on Block of 32 Weight Parameters
np.random.seed(42)
true_weights = np.random.normal(0, 0.2, 32)

scale, packed_data = quantize_block_q4_0(true_weights)
reconstructed = dequantize_block_q4_0(scale, packed_data)

original_bytes = true_weights.nbytes # 32 * 4 = 128 bytes
compressed_bytes = len(packed_data) + 4 # 16 bytes payload + 4 bytes scale = 20 bytes

print("--- GGUF Block Quantization (Q4_0) ---")
print(f"Original 32-Float Block RAM:    {original_bytes} bytes")
print(f"Q4_0 Quantized Block RAM:       {compressed_bytes} bytes")
print(f"Memory Compression Ratio:       {(1 - compressed_bytes / original_bytes) * 100:.1f}%")
print(f"Mean Absolute Dequant Error:    {np.mean(np.abs(true_weights - reconstructed)):.4f}")

### 3. Step-by-Step Code Walkthrough
1. **Block Partitioning:** Slices matrices into blocks of 32 numbers to limit local quantization error.
2. **Nibble Packing:** Packs two 4-bit weights into a single 8-bit byte.
3. **Fast CPU SIMD Dequantization:** Can be dequantized via AVX2 / ARM NEON vector instructions on consumer CPUs.

---

### 4. Industry Applications & Trade-offs
- **On-Device Inference (Ollama, LM Studio):** Running 8B to 70B models locally on consumer PCs and laptops.
- **Trade-off:** 4-bit quantization degrades performance on difficult reasoning benchmarks unless k-quant variants (Q4_K_M) are used.

---

### 5. Top Interview Questions & Answers
**Q1: What is the primary architectural improvement of GGUF over its predecessor GGML?**
> **Answer:** GGML embedded fixed assumptions about model architectures in C++ code. GGUF provides a flexible binary key-value metadata store, allowing new model architectures (like MoE and Mamba) to be loaded without breaking backward compatibility.

**Q2: What is the difference between Q4_0 and Q4_K_M quantizations?**
> **Answer:** Q4_0 uses a single scale per 32-weight block uniformly across all layers. Q4_K_M uses variable bit allocations across layers (e.g. 6-bit for critical attention layers, 4-bit for feedforward networks) to preserve model quality.